# Data Validation


In [1]:
import pandas as pd

In [4]:
df=pd.read_csv("../Data/final/environment_data.csv")

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.isnull().sum()

City                       0
Date                       0
PM2.5                      0
PM10                       0
O3                         0
NO2                        0
CO                         0
SO2                        0
Green_Space                0
Temperature_mean           0
Temperature_max            0
Humidity                   0
Wind_speed                 0
Pressure                   0
Electricity Consumption    0
Source                     0
isWeekend                  0
Season                     0
dtype: int64

In [7]:
df.describe()

,PM2.5,PM10,O3,NO2,CO,SO2,Green_Space,Temperature_mean,Temperature_max,Humidity,Wind_speed,Pressure,Electricity Consumption,isWeekend
count,2548.000000,2548.000000,2548.000000,2548.000000,2548.000000,2548.000000,2548.000000,2548.000000,2548.000000,2548.000000,2548.000000,2548.000000,2.548000e+03,2548.000000
mean,20.417589,37.057589,54.534161,33.155532,338.559409,14.461962,105.139717,19.034251,23.260204,64.620222,13.672576,984.927908,7.302820e+08,0.286892
std,14.069021,36.686546,26.078559,23.776493,219.605313,17.562906,46.006487,8.417289,9.220257,15.053140,5.369632,59.903467,7.652178e+08,0.452399
min,2.887500,4.166667,1.416667,1.841667,111.125000,0.200000,51.000000,-8.681818,-3.000000,16.560000,2.000000,835.370000,4.856317e+07,0.000000
25%,9.403125,14.604167,37.541667,15.887500,204.812500,3.207292,53.000000,12.661458,16.000000,54.520833,9.958333,997.455260,3.125282e+08,0.000000
50%,16.943750,25.450000,53.250000,27.539583,278.895833,6.202083,120.000000,19.465000,24.000000,65.000000,12.750000,1007.051963,4.133552e+08,0.000000
75%,26.831250,43.475000,69.354167,44.290625,400.572917,22.671875,162.000000,24.333333,29.000000,76.306738,16.196314,1014.756474,8.133111e+08,1.000000
max,91.745833,348.716667,220.333333,180.391667,2661.125000,157.254167,164.000000,40.625000,71.000000,97.625000,46.916667,1041.341875,3.015648e+09,1.000000


In [66]:
def detect_outliers_iqr(df, city=None):
    if city is not None:
        df = df[df["City"] == city]
    
    outlier_summary = []
    outlier_rows = pd.DataFrame()

    for col in df.select_dtypes(include=['float64', 'int64']).columns:
        if col == "isWeekend":
            continue

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        outliers = df[(df[col] < lower) | (df[col] > upper)]
        outliers["Outlier_Feature"] = col   
        outlier_rows = pd.concat([outlier_rows, outliers])
        
        outlier_summary.append({
            "Feature": col,
            "Outliers_Count": len(outliers),
            "Lower_Bound": round(lower, 2),
            "Upper_Bound": round(upper, 2)
        })
    
    return pd.DataFrame(outlier_summary), outlier_rows

In [67]:
summary, outlier_rows = detect_outliers_iqr(df, city="Cairo")

print(summary.sort_values(by="Outliers_Count", ascending=False))

                    Feature  Outliers_Count   Lower_Bound   Upper_Bound
9                  Humidity              37  3.146000e+01  7.696000e+01
1                      PM10              26  3.630000e+00  8.074000e+01
5                       SO2              24  6.610000e+00  5.521000e+01
4                        CO              21  6.840000e+01  4.755600e+02
0                     PM2.5              19  9.020000e+00  4.255000e+01
3                       NO2               6 -6.600000e-01  6.076000e+01
10               Wind_speed               5  1.920000e+00  2.358000e+01
2                        O3               0 -5.190000e+00  1.316500e+02
6               Green_Space               0  5.300000e+01  5.300000e+01
8           Temperature_max               0  6.500000e+00  5.050000e+01
7          Temperature_mean               0  3.890000e+00  4.314000e+01
11                 Pressure               0  9.858900e+02  1.013170e+03
12  Electricity Consumption               0  2.567566e+08  5.052

C:\Users\AL SAAD NASR CITY\AppData\Local\Temp\ipykernel_18280\2501839798.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outliers["Outlier_Feature"] = col
C:\Users\AL SAAD NASR CITY\AppData\Local\Temp\ipykernel_18280\2501839798.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outliers["Outlier_Feature"] = col
C:\Users\AL SAAD NASR CITY\AppData\Local\Temp\ipykernel_18280\2501839798.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_i

In [68]:
outlier_rows["Outlier_Feature"].value_counts()

Outlier_Feature
Humidity      37
PM10          26
SO2           24
CO            21
PM2.5         19
NO2            6
Wind_speed     5
Name: count, dtype: int64

outliers are mainly concentrated in humidity and pollution-related features, indicating that both atmospheric conditions and air pollution levels exhibit high variability in Cairo.

In [69]:
outlier_rows["Date"] = pd.to_datetime(outlier_rows["Date"])
outlier_rows["Date"].dt.month.value_counts()

Date
12    33
1     22
5     16
11    15
2     11
3     11
6      9
4      8
9      8
10     4
8      1
Name: count, dtype: int64

Outliers are more frequent during winter months (December and January), indicating seasonal patterns in environmental variability.

In [63]:
outlier_rows.groupby("Outlier_Feature")[[
    "Wind_speed", "Humidity", "Temperature_mean"
]].mean()
# when outiler happens in each feature

,Wind_speed,Humidity,Temperature_mean
Outlier_Feature,,,
CO,6.253722,56.001994,18.618416
Humidity,13.662575,43.599787,25.382899
NO2,4.985813,50.815944,18.530240
PM10,17.194662,43.246816,24.382471
PM2.5,13.856891,51.696884,23.018013
SO2,5.899297,56.156826,18.349083
Wind_speed,24.995946,36.252302,22.564338


Gaseous pollutants (CO, SO2, NO2) tend to spike under low wind conditions, while particulate matter (PM10) increases with higher wind speeds due to dust resuspension.

In [62]:
normal_rows = df[~df.index.isin(outlier_rows.index)]

outliers_mean = outlier_rows.mean(numeric_only=True)
normal_mean = normal_rows.mean(numeric_only=True)

comparison = pd.DataFrame({
    "Outliers": outliers_mean,
    "Normal": normal_mean
})

comparison

,Outliers,Normal
PM2.5,3.742464e+01,1.991172e+01
PM10,7.996416e+01,3.583327e+01
O3,4.410356e+01,5.483431e+01
NO2,4.182107e+01,3.291811e+01
CO,3.838454e+02,3.378126e+02
SO2,4.818270e+01,1.344622e+01
Green_Space,5.300000e+01,1.069172e+02
Temperature_mean,2.221610e+01,1.891910e+01
Temperature_max,2.681159e+01,2.312946e+01
Humidity,4.876676e+01,6.514026e+01


While pollution levels are higher during outlier events, meteorological conditions such as wind speed and humidity do not differ significantly from normal days, suggesting that pollution spikes are not driven by a single factor but rather a combination of conditions.

#dubai


In [70]:
summary, outlier_rows = detect_outliers_iqr(df, city="Dubai")


print(summary.sort_values(by="Outliers_Count", ascending=False))

                    Feature  Outliers_Count   Lower_Bound   Upper_Bound
3                       NO2              17 -1.312000e+01  1.210800e+02
10               Wind_speed              15  6.620000e+00  1.923000e+01
2                        O3              10 -1.546000e+01  1.648800e+02
1                      PM10              10 -3.671000e+01  2.260400e+02
4                        CO               5  6.912000e+01  8.461200e+02
9                  Humidity               3  2.074000e+01  8.309000e+01
0                     PM2.5               2 -3.110000e+00  8.277000e+01
5                       SO2               1  6.200000e-01  4.108000e+01
6               Green_Space               0  5.100000e+01  5.100000e+01
8           Temperature_max               0  1.250000e+01  5.650000e+01
7          Temperature_mean               0  9.060000e+00  5.123000e+01
11                 Pressure               0  9.820000e+02  1.034290e+03
12  Electricity Consumption               0  1.981287e+08  4.150

C:\Users\AL SAAD NASR CITY\AppData\Local\Temp\ipykernel_18280\2501839798.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outliers["Outlier_Feature"] = col
C:\Users\AL SAAD NASR CITY\AppData\Local\Temp\ipykernel_18280\2501839798.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outliers["Outlier_Feature"] = col
C:\Users\AL SAAD NASR CITY\AppData\Local\Temp\ipykernel_18280\2501839798.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_i

In [71]:
outlier_rows["Outlier_Feature"].value_counts()

Outlier_Feature
NO2           17
Wind_speed    15
O3            10
PM10          10
CO             5
Humidity       3
PM2.5          2
SO2            1
Name: count, dtype: int64

Outliers in Dubai are dominated by NO2, wind speed, and ozone, indicating that pollution is influenced by traffic emissions, strong wind conditions, and intense solar radiation.

In [72]:
outlier_rows.groupby("Outlier_Feature")[["Wind_speed","Humidity","Temperature_mean"]].mean()

,Wind_speed,Humidity,Temperature_mean
Outlier_Feature,,,
CO,11.050641,46.989103,25.290385
Humidity,15.358750,41.099861,30.367083
NO2,11.616433,43.534605,27.504872
O3,13.362104,47.778947,37.766088
PM10,14.602212,41.985991,38.683902
PM2.5,13.250000,34.041667,39.541667
SO2,12.400000,54.960000,32.760000
Wind_speed,22.451347,54.673497,25.477293


In Dubai, pollution spikes are primarily driven by high temperatures and solar radiation, rather than low wind conditions. This leads to increased ozone formation and dust-related pollution, reflecting the impact of desert climate and intense sunlight.

In [73]:
normal_rows = df[~df.index.isin(outlier_rows.index)]

outliers_mean = outlier_rows.mean(numeric_only=True)
normal_mean = normal_rows.mean(numeric_only=True)

comparison = pd.DataFrame({
    "Outliers": outliers_mean,
    "Normal": normal_mean
})

comparison

,Outliers,Normal
PM2.5,5.216819e+01,1.978536e+01
PM10,1.358855e+02,3.503105e+01
O3,8.900794e+01,5.377316e+01
NO2,7.334974e+01,3.237277e+01
CO,5.660007e+02,3.343435e+02
SO2,2.309550e+01,1.429777e+01
Green_Space,5.100000e+01,1.062898e+02
Temperature_mean,3.085141e+01,1.877936e+01
Temperature_max,3.547619e+01,2.299840e+01
Humidity,4.665283e+01,6.501062e+01


In Dubai, pollution spikes are primarily driven by high temperatures, dry conditions, and increased wind activity, which together promote ozone formation and dust resuspension. These events are not directly linked to electricity consumption but are strongly influenced by environmental and climatic factors.